In [1]:
!pip install pandas praat-parselmouth tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 86.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip /content/drive/MyDrive/project/ru-fr_interference.zip -d /content/data

Streaming output truncated to the last 5000 lines.
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp58.txt  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp14.txt  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp20.wav  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp30.TextGrid  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp16.txt  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp47.txt  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp75.wav  
  inflating: /content/data/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only/AN/an_fra_list1_FRcorp35_words.

delete everything before this cell below then save as .py file for dvc

In [49]:
import os
import pandas as pd
import parselmouth
from tqdm import tqdm

os.environ["DATA_DIR"] = "/content/data/ru-fr_interference/2"
BASE_DIR = os.getenv("DATA_DIR", "./data")
# Path to the folder that contains one folder per speaker
DATA_DIR = os.path.join(BASE_DIR, "wav_et_textgrids", "FRcorp_textgrids_only")
META_PATH = os.path.join(BASE_DIR, "metadata_RUFR.csv")
# Path where we save the final phoneme table
OUTPUT_PATH = os.path.join(BASE_DIR, "phoneme_table.csv")

# We store each phoneme token as one row in this list
rows = []

In [51]:
# Loop over speaker folders
for speaker_id in os.listdir(DATA_DIR):
    speaker_path = os.path.join(DATA_DIR, speaker_id)
    # Skip anything that is not a folder
    if not os.path.isdir(speaker_path):
        continue
    # Loop over files inside one speaker folder
    for file in os.listdir(speaker_path):
        # We only need TextGrid files here
        if not file.endswith(".TextGrid"):
            continue
        tg_path = os.path.join(speaker_path, file) # Full path to the TextGrid file
        wav_path = tg_path.replace(".TextGrid", ".wav") # The wav file should have the same name as the TextGrid file
        if not os.path.exists(wav_path):
            continue
        fname = file.replace(".TextGrid", "") # Remove the file extension and Split the filename into parts
        parts = fname.split("_")
        lang = parts[1] # parts[1] is the language group: fra or rus
        sentence_id = parts[-1] # The last part gives the sentence id
        L1 = "L1" if lang == "fra" else "L2"
        rep = parts[2] # repetition index e.g: list1
        rep_idx = int(rep.replace("list", ""))
        # Read the TextGrid as a normal text file
        with open(tg_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        # We will collect only intervals from the phones tier
        in_phones_tier = False
        current_interval = {}
        phoneme_list = []

        for line in lines:
            line = line.strip()
            if line.startswith('name = "phones"'): # Detect the start of the phones tier
                in_phones_tier = True
                continue
            if in_phones_tier and line.startswith("item ["): # in case there are more than two tiers skip them
                break
            # Read interval information inside phones tier
            if in_phones_tier:
                if line.startswith("xmin ="):
                    current_interval["onset"] = float(line.split("=")[1].strip())
                elif line.startswith("xmax ="):
                    current_interval["offset"] = float(line.split("=")[1].strip())
                elif line.startswith("text ="):
                    label = line.split("=", 1)[1].strip().strip('"')
                    # Skip empty and silence intervals
                    if label not in ["", "sil", "sp", "spn"] and "..." not in label and len(label) <= 3:
                        current_interval["label"] = label
                        phoneme_list.append(current_interval.copy())
                    current_interval = {} # just to be safe, we reset
        # creating a list of all phonemes
        for p in phoneme_list:
            rows.append({
                "speaker_id": speaker_id,
                "sentence_id": sentence_id,
                "repetition": rep_idx,
                "filename": fname,
                "lang": lang,
                "L1": L1,
                "label": p["label"],
                "onset": p["onset"],
                "duration": p["offset"] - p["onset"],
                "offset": p["offset"],
                "wav_path": wav_path
            })

In [52]:
df = pd.DataFrame(rows)

In [53]:
df

,speaker_id,sentence_id,repetition,filename,lang,L1,label,onset,duration,offset,wav_path
0,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,d,0.45231,0.10769,0.56000,/content/data/ru-fr_interference/2/wav_et_text...
1,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,i,0.56000,0.18000,0.74000,/content/data/ru-fr_interference/2/wav_et_text...
2,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,s,0.74000,0.14000,0.88000,/content/data/ru-fr_interference/2/wav_et_text...
3,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,e,0.88000,0.12142,1.00142,/content/data/ru-fr_interference/2/wav_et_text...
4,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,ʁ,1.00142,0.07858,1.08000,/content/data/ru-fr_interference/2/wav_et_text...
...,...,...,...,...,...,...,...,...,...,...,...
45187,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,w,1.67000,0.03000,1.70000,/content/data/ru-fr_interference/2/wav_et_text...
45188,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,a,1.70000,0.06000,1.76000,/content/data/ru-fr_interference/2/wav_et_text...
45189,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,f,1.76000,0.12000,1.88000,/content/data/ru-fr_interference/2/wav_et_text...
45190,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,w,1.88000,0.04000,1.92000,/content/data/ru-fr_interference/2/wav_et_text...


In [54]:
meta = pd.read_csv(META_PATH, sep=";")
meta = meta.rename(columns={
    "spk": "speaker_id",
    "Gender": "gender",
    "Age": "age"
})
df = df.merge(meta[["speaker_id", "gender", "age"]], on="speaker_id", how="left")

In [55]:
df

,speaker_id,sentence_id,repetition,filename,lang,L1,label,onset,duration,offset,wav_path,gender,age
0,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,d,0.45231,0.10769,0.56000,/content/data/ru-fr_interference/2/wav_et_text...,f,22
1,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,i,0.56000,0.18000,0.74000,/content/data/ru-fr_interference/2/wav_et_text...,f,22
2,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,s,0.74000,0.14000,0.88000,/content/data/ru-fr_interference/2/wav_et_text...,f,22
3,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,e,0.88000,0.12142,1.00142,/content/data/ru-fr_interference/2/wav_et_text...,f,22
4,AB,FRcorp69,1,ab_rus_list1_FRcorp69,rus,L2,ʁ,1.00142,0.07858,1.08000,/content/data/ru-fr_interference/2/wav_et_text...,f,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45187,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,w,1.67000,0.03000,1.70000,/content/data/ru-fr_interference/2/wav_et_text...,m,23
45188,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,a,1.70000,0.06000,1.76000,/content/data/ru-fr_interference/2/wav_et_text...,m,23
45189,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,f,1.76000,0.12000,1.88000,/content/data/ru-fr_interference/2/wav_et_text...,m,23
45190,CB,FRcorp60,1,cb_fra_list1_FRcorp60,fra,L1,w,1.88000,0.04000,1.92000,/content/data/ru-fr_interference/2/wav_et_text...,m,23


In [23]:
df.to_csv(OUTPUT_PATH, index=False)